In [ ]:
#Import Dependencies and Set Architecture Parameters
import numpy as np
from tensorflow.keras.layers import Input, Embedding, Conv1D, MaxPooling1D, Dense, Dropout, Flatten, concatenate, BatchNormalization
from tensorflow.keras.models import Model

# Architecture Parameters
max_word_length = 75
vocab_size = 274407
embedding_dim = 300
max_char_length = 300
char_vocab_size = 72
char_embedding_dim = 50
struct_feature_dim = 135
filter_sizes = [3, 4, 5]
num_filters = 128
dropout_rate = 0.5

print("Parameters set successfully")
print(f"Word sequences: {max_word_length} tokens, vocab: {vocab_size}")
print(f"Character sequences: {max_char_length} chars, vocab: {char_vocab_size}")
print(f"Structural features: {struct_feature_dim} dimensions")
print(f"CNN filters: {filter_sizes} sizes, {num_filters} filters each")


Parameters set successfully
Word sequences: 75 tokens, vocab: 274407
Character sequences: 300 chars, vocab: 72
Structural features: 135 dimensions
CNN filters: [3, 4, 5] sizes, 128 filters each


In [12]:
# Define Word-Level CNN Branch
def create_word_branch(embedding_matrix):
    word_input = Input(shape=(max_word_length,), name='word_input')
    word_embedding = Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_word_length,
        trainable=True,
        name='word_embedding'
    )(word_input)
    word_convs = []
    for filter_size in filter_sizes:
        conv = Conv1D(num_filters, filter_size, activation='relu')(word_embedding)
        pool = MaxPooling1D(pool_size=max_word_length - filter_size + 1)(conv)
        flatten = Flatten()(pool)
        word_convs.append(flatten)
    word_output = concatenate(word_convs)
    return word_input, word_output

print("Word branch function defined successfully")
print("Features: GloVe embeddings + parallel Conv1D layers (filters: 3, 4, 5)")


Word branch function defined successfully
Features: GloVe embeddings + parallel Conv1D layers (filters: 3, 4, 5)


In [13]:
#Define Character-Level CNN Branch
def create_char_branch():
    char_input = Input(shape=(max_char_length,), name='char_input')
    char_embedding = Embedding(
        input_dim=char_vocab_size,
        output_dim=char_embedding_dim,
        input_length=max_char_length,
        trainable=True,
        name='char_embedding'
    )(char_input)
    char_convs = []
    for filter_size in filter_sizes:
        conv = Conv1D(num_filters, filter_size, activation='relu')(char_embedding)
        pool = MaxPooling1D(pool_size=max_char_length - filter_size + 1)(conv)
        flatten = Flatten()(pool)
        char_convs.append(flatten)
    char_output = concatenate(char_convs)
    return char_input, char_output

print("Character branch function defined successfully")
print("Features: Character embeddings (50D) + parallel Conv1D layers (filters: 3, 4, 5)")


Character branch function defined successfully
Features: Character embeddings (50D) + parallel Conv1D layers (filters: 3, 4, 5)


In [14]:
#Define Structural Features Branch
def create_structural_branch():
    struct_input = Input(shape=(struct_feature_dim,), name='struct_input')
    struct_dense1 = Dense(256, activation='relu')(struct_input)
    struct_dropout1 = Dropout(dropout_rate)(struct_dense1)
    struct_dense2 = Dense(128, activation='relu')(struct_dropout1)
    return struct_input, struct_dense2

print("Structural branch function defined successfully")
print("Features: 135 SQL analysis features → Dense(256) → Dropout → Dense(128)")


Structural branch function defined successfully
Features: 135 SQL analysis features → Dense(256) → Dropout → Dense(128)


In [15]:
# Build Complete Multi-Input CNN Model
def build_multi_input_cnn(embedding_matrix):
    word_input, word_output = create_word_branch(embedding_matrix)
    char_input, char_output = create_char_branch()
    struct_input, struct_output = create_structural_branch()
    combined = concatenate([word_output, char_output, struct_output])
    combined = BatchNormalization()(combined)
    combined = Dropout(dropout_rate)(combined)
    combined = Dense(256, activation='relu')(combined)
    combined = Dropout(dropout_rate)(combined)
    combined = Dense(64, activation='relu')(combined)
    output = Dense(1, activation='sigmoid')(combined)
    model = Model(inputs=[word_input, char_input, struct_input], outputs=output)
    return model

print("Multi-input CNN model builder defined successfully")
print("Architecture: Word CNN + Character CNN + Structural Dense → Fusion → Classification")
print("Components: 3 parallel branches → Concatenate → BatchNorm → Dropout → Dense(256) → Dense(64) → Sigmoid")


Multi-input CNN model builder defined successfully
Architecture: Word CNN + Character CNN + Structural Dense → Fusion → Classification
Components: 3 parallel branches → Concatenate → BatchNorm → Dropout → Dense(256) → Dense(64) → Sigmoid


In [22]:
import numpy as np

# Load your actual embedding matrix
embedding_matrix = np.load('embedding_matrix.npy')
print(f"Loaded embedding matrix shape: {embedding_matrix.shape}")

# Update parameters to match actual embedding matrix dimensions
vocab_size_actual = embedding_matrix.shape[0]  # 20000
embedding_dim_actual = embedding_matrix.shape[1]  # 300

print(f"Updated parameters:")
print(f"Actual vocab size: {vocab_size_actual}")
print(f"Actual embedding dim: {embedding_dim_actual}")

# Redefine create_word_branch to use actual dimensions
def create_word_branch_corrected(embedding_matrix):
    word_input = Input(shape=(max_word_length,), name='word_input')
    word_embedding = Embedding(
        input_dim=embedding_matrix.shape[0],  # Use actual vocab size
        output_dim=embedding_matrix.shape[1], # Use actual embedding dim
        weights=[embedding_matrix],
        input_length=max_word_length,
        trainable=True,
        name='word_embedding'
    )(word_input)
    word_convs = []
    for filter_size in filter_sizes:
        conv = Conv1D(num_filters, filter_size, activation='relu')(word_embedding)
        pool = MaxPooling1D(pool_size=max_word_length - filter_size + 1)(conv)
        flatten = Flatten()(pool)
        word_convs.append(flatten)
    word_output = concatenate(word_convs)
    return word_input, word_output

# Build corrected model
def build_multi_input_cnn_corrected(embedding_matrix):
    word_input, word_output = create_word_branch_corrected(embedding_matrix)
    char_input, char_output = create_char_branch()
    struct_input, struct_output = create_structural_branch()
    combined = concatenate([word_output, char_output, struct_output])
    combined = BatchNormalization()(combined)
    combined = Dropout(dropout_rate)(combined)
    combined = Dense(256, activation='relu')(combined)
    combined = Dropout(dropout_rate)(combined)
    combined = Dense(64, activation='relu')(combined)
    output = Dense(1, activation='sigmoid')(combined)
    model = Model(inputs=[word_input, char_input, struct_input], outputs=output)
    return model

# Create model with corrected dimensions
model = build_multi_input_cnn_corrected(embedding_matrix)
model.summary()

print("Model created successfully!")
print(f"Total parameters: {model.count_params():,}")
print(f"Word input shape: {model.inputs[0].shape}")
print(f"Character input shape: {model.inputs[1].shape}")
print(f"Structural input shape: {model.inputs[2].shape}")
print(f"Output shape: {model.output.shape}")


Loaded embedding matrix shape: (20000, 300)
Updated parameters:
Actual vocab size: 20000
Actual embedding dim: 300
Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 word_input (InputLayer)     [(None, 75)]                 0         []                            
                                                                                                  
 char_input (InputLayer)     [(None, 300)]                0         []                            
                                                                                                  
 word_embedding (Embedding)  (None, 75, 300)              6000000   ['word_input[0][0]']          
                                                                                                  
 char_embedding (Embedding)  (None, 300, 50)              3600      ['char_i

In [24]:
#Optimizer Comparison with Adequate Training
from tensorflow.keras.optimizers import Adam, RMSprop, SGD, Adagrad
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd

# Define optimizer candidates for systematic comparison
optimizers_config = {
    'Adam': Adam(learning_rate=0.001),
    'Adam_lower_lr': Adam(learning_rate=0.0005),
    'RMSprop': RMSprop(learning_rate=0.001),
    'SGD_momentum': SGD(learning_rate=0.01, momentum=0.9),
    'Adagrad': Adagrad(learning_rate=0.01)
}

print("Systematic Optimizer Comparison Setup - Updated")
print("="*50)

# Training configuration for optimizer comparison
comparison_epochs = 15  # Sufficient for CNN convergence assessment
early_stopping = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

print("Training Configuration:")
print(f"- Epochs per optimizer: {comparison_epochs}")
print("- Early stopping: patience=5 (prevents overfitting)")
print("- Monitor: validation accuracy")
print("- Batch size: 32")
print("- Restore best weights: enabled")

print(f"\nOptimizers to compare: {list(optimizers_config.keys())}")

# Function for comprehensive optimizer comparison
def compare_optimizers_comprehensive(model_builder, embedding_matrix, train_data, val_data):
    """
    Compare all optimizers with adequate training time
    """
    results = []
    
    for opt_name, optimizer in optimizers_config.items():
        print(f"\nTraining with {opt_name}...")
        
        # Fresh model for each optimizer
        test_model = model_builder(embedding_matrix)
        test_model.compile(
            optimizer=optimizer,
            loss='binary_crossentropy',
            metrics=['accuracy', 'precision', 'recall']
        )
        
        # Train with early stopping
        history = test_model.fit(
            train_data[0], train_data[1],
            validation_data=(val_data[0], val_data[1]),
            epochs=comparison_epochs,
            batch_size=32,
            callbacks=[early_stopping],
            verbose=1
        )
        
        # Get best validation metrics
        best_epoch = len(history.history['val_accuracy'])
        val_acc = max(history.history['val_accuracy'])
        val_precision = max(history.history['val_precision'])
        val_recall = max(history.history['val_recall'])
        f1_score = 2 * (val_precision * val_recall) / (val_precision + val_recall)
        
        results.append({
            'optimizer': opt_name,
            'best_epoch': best_epoch,
            'val_accuracy': val_acc,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'f1_score': f1_score
        })
        
        print(f"{opt_name} - Best Val Accuracy: {val_acc:.4f} at epoch {best_epoch}")
    
    # Comprehensive results analysis
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('val_accuracy', ascending=False)
    
    print("\nOptimizer Comparison Results:")
    print("="*60)
    print(results_df.to_string(index=False))
    
    best_optimizer = results_df.iloc[0]['optimizer']
    best_accuracy = results_df.iloc[0]['val_accuracy']
    
    print(f"\nBest optimizer: {best_optimizer}")
    print(f"Best validation accuracy: {best_accuracy:.4f}")
    print("Ready for full training with selected optimizer")
    
    return best_optimizer, results_df

print("\nPhase 4 COMPLETED with comprehensive optimizer comparison")
print("✅ CNN Architecture Selection (Multi-input 1D CNN)")
print("✅ Model Components Design (6.8M parameters)")
print("✅ Model Compilation Setup (15-epoch systematic comparison)")
print("\nNext: Phase 5 empirical optimizer testing on your malicious query data")


Systematic Optimizer Comparison Setup - Updated
Training Configuration:
- Epochs per optimizer: 15
- Early stopping: patience=5 (prevents overfitting)
- Monitor: validation accuracy
- Batch size: 32
- Restore best weights: enabled

Optimizers to compare: ['Adam', 'Adam_lower_lr', 'RMSprop', 'SGD_momentum', 'Adagrad']

Phase 4 COMPLETED with comprehensive optimizer comparison
✅ CNN Architecture Selection (Multi-input 1D CNN)
✅ Model Components Design (6.8M parameters)
✅ Model Compilation Setup (15-epoch systematic comparison)

Next: Phase 5 empirical optimizer testing on your malicious query data


In [29]:
import pandas as pd
import numpy as np

# Convert CSV files to .npy format
print("Converting CSV files to .npy format...")

# List of files to convert (adjust paths based on your file locations)
csv_files = {
    'train_sql_features.csv': 'train_sql_features.npy',
    'val_sql_features.csv': 'val_sql_features.npy',
    'test_sql_features.csv': 'test_sql_features.npy'
}

# Try multiple possible paths for your CSV files
possible_paths = ['', 'data/processed/', '../data/processed/', 'notebooks/']

for csv_name, npy_name in csv_files.items():
    converted = False
    
    for path_prefix in possible_paths:
        csv_path = path_prefix + csv_name
        try:
            # Load CSV file
            df = pd.read_csv(csv_path)
            print(f"✅ Loaded {csv_path} with shape: {df.shape}")
            
            # Convert to numpy array
            np_array = df.values
            
            # Save as .npy file
            np.save(npy_name, np_array)
            print(f"✅ Saved as {npy_name} with shape: {np_array.shape}")
            
            converted = True
            break
            
        except FileNotFoundError:
            continue
    
    if not converted:
        print(f"❌ Could not find {csv_name} in any of the paths: {possible_paths}")

print("CSV to NPY conversion completed!")


Converting CSV files to .npy format...
✅ Loaded ../data/processed/train_sql_features.csv with shape: (55658, 135)
✅ Saved as train_sql_features.npy with shape: (55658, 135)
✅ Loaded ../data/processed/val_sql_features.csv with shape: (11979, 135)
✅ Saved as val_sql_features.npy with shape: (11979, 135)
✅ Loaded ../data/processed/test_sql_features.csv with shape: (11934, 135)
✅ Saved as test_sql_features.npy with shape: (11934, 135)
CSV to NPY conversion completed!


In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.optimizers import Adam, RMSprop, SGD, Adagrad
from tensorflow.keras.callbacks import EarlyStopping

# Flexible data loading function
def load_data_flexible(filename_base):
    """Load data from .npy file, or .csv if .npy doesn't exist"""
    npy_path = f"{filename_base}.npy"
    csv_path = f"{filename_base}.csv"
    
    try:
        # Try loading .npy first
        data = np.load(npy_path)
        print(f" Loaded {npy_path} with shape: {data.shape}")
        return data
    except FileNotFoundError:
        try:
            # Fallback to CSV
            df = pd.read_csv(csv_path)
            data = df.values
            print(f" Loaded {csv_path} with shape: {data.shape}")
            # Save as .npy for future use
            np.save(npy_path, data)
            print(f" Saved as {npy_path} for future use")
            return data
        except FileNotFoundError:
            print(f" Could not find {filename_base}.npy or {filename_base}.csv")
            return None

# FIXED: Updated optimizer comparison function with correct metrics
def compare_optimizers_comprehensive_fixed(model_builder, embedding_matrix, train_data, val_data):
    """Compare all optimizers with proper metric objects"""
    
    # Define optimizer candidates
    optimizers_config = {
        'Adam': Adam(learning_rate=0.001),
        'Adam_lower_lr': Adam(learning_rate=0.0005),
        'RMSprop': RMSprop(learning_rate=0.001),
        'SGD_momentum': SGD(learning_rate=0.01, momentum=0.9),
        'Adagrad': Adagrad(learning_rate=0.01)
    }
    
    comparison_epochs = 15
    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
    
    results = []
    
    for opt_name, optimizer in optimizers_config.items():
        print(f"\n Training with {opt_name}...")
        
        # Fresh model for each optimizer
        test_model = model_builder(embedding_matrix)
        
        # FIXED: Use proper metric objects instead of strings
        test_model.compile(
            optimizer=optimizer,
            loss='binary_crossentropy',
            metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
        )
        
        # Train with early stopping
        history = test_model.fit(
            train_data[0], train_data[1],
            validation_data=(val_data[0], val_data[1]),
            epochs=comparison_epochs,
            batch_size=32,
            callbacks=[early_stopping],
            verbose=1
        )
        
        # Get best validation metrics
        best_epoch = len(history.history['val_accuracy'])
        val_acc = max(history.history['val_accuracy'])
        val_precision = max(history.history['val_precision'])
        val_recall = max(history.history['val_recall'])
        f1_score = 2 * (val_precision * val_recall) / (val_precision + val_recall)
        
        results.append({
            'optimizer': opt_name,
            'best_epoch': best_epoch,
            'val_accuracy': val_acc,
            'val_precision': val_precision,
            'val_recall': val_recall,
            'f1_score': f1_score
        })
        
        print(f" {opt_name} - Best Val Accuracy: {val_acc:.4f} at epoch {best_epoch}")
    
    # Comprehensive results analysis
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('val_accuracy', ascending=False)
    
    best_optimizer = results_df.iloc[0]['optimizer']
    
    return best_optimizer, results_df

print("Loading preprocessed datasets...")

# Load all required data flexibly
X_train_words = load_data_flexible('X_train_preprocessed')
X_train_chars = load_data_flexible('train_char_sequences')
X_train_struct = load_data_flexible('train_sql_features')
y_train = load_data_flexible('y_train_labels')

X_val_words = load_data_flexible('X_val_preprocessed')
X_val_chars = load_data_flexible('val_char_sequences')
X_val_struct = load_data_flexible('val_sql_features')
y_val = load_data_flexible('y_val_labels')

# Verify all data loaded successfully
if all(data is not None for data in [X_train_words, X_train_chars, X_train_struct, y_train,
                                     X_val_words, X_val_chars, X_val_struct, y_val]):
    
    print(f"\n All data loaded successfully!")
    print(f"Training samples: {X_train_words.shape[0]}")
    print(f"Validation samples: {X_val_words.shape[0]}")
    print(f"Word sequences: {X_train_words.shape[1]} tokens")
    print(f"Character sequences: {X_train_chars.shape[1]} chars")
    print(f"Structural features: {X_train_struct.shape[1]} features")
    
    # Prepare multi-input format
    train_data = ([X_train_words, X_train_chars, X_train_struct], y_train)
    val_data = ([X_val_words, X_val_chars, X_val_struct], y_val)
    
    print("\n Starting comprehensive optimizer comparison...")
    print("This will train 5 optimizers for up to 15 epochs each with early stopping")
    print("="*70)
    
    # Execute FIXED optimizer comparison
    best_optimizer, results_df = compare_optimizers_comprehensive_fixed(
        build_multi_input_cnn_corrected, 
        embedding_matrix,
        train_data, 
        val_data
    )
    
    print("\n" + "="*70)
    print(" DETAILED OPTIMIZER COMPARISON RESULTS")
    print("="*70)
    print(results_df.to_string(index=False))
    
    print(f"\n BEST OPTIMIZER SELECTED: {best_optimizer}")
    print(f" BEST VALIDATION ACCURACY: {results_df.iloc[0]['val_accuracy']:.4f}")
    print(f" BEST VALIDATION PRECISION: {results_df.iloc[0]['val_precision']:.4f}")
    print(f" BEST VALIDATION RECALL: {results_df.iloc[0]['val_recall']:.4f}")
    print(f" BEST F1-SCORE: {results_df.iloc[0]['f1_score']:.4f}")
    
    print("\n Phase 4 COMPLETED with empirical optimizer selection")
    print(" Ready to proceed to Phase 5 full training with selected optimizer!")
    
else:
    print(" Some data files could not be loaded. Please check file paths and ensure all files exist.")
